# Additional End of week Exercise - week 2

In [7]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from artists import tools, handle_tool_calls
from artists_met import get_image

In [61]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if not(openai_api_key):
    print("OpenAI API Key not set")

LAMA_MODEL = "llama3.2"
DEEPSEEK_MODEL = "deepseek-r1:1.5b"
GPT_MODEL = "gpt-4.1-mini"
OLLAMA_BASE_URL = "http://localhost:11434"

openai = OpenAI()
hasOllama = requests.get(OLLAMA_BASE_URL).content
print(f"Ollama is running: {bool(hasOllama)}")

ollama = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key='ollama')

Ollama is running: True


In [76]:
system_message = """You are a helpful guide, working for the Metropolitan Museum of Art in New York. You provide information about artists and shows images of their artworks. 
The tools that are provided to you in this chat give you access to a dataset with information about artists, including their names, the period in which he/she lived, and the collections their work belongs to. Also from an artist's name, you can get a random artwork from that artist, including the title and an image of the artwork.
If you don't find an artist by the name the user provided, first try to use the get_artist_suggestions to check if you can ask if the user meant one of the suggested artists. If you find a suggestion that matches the user's intent, you can ask the user if they meant that artist. If the user confirms, you can then use the get_random_artwork_from_artist tool to get information about that artist and their artworks.
Keep in mind that the user does not always asks for an artist but also can start a normal social conversation, like a greeting or friendly talk, etcera.
From the collections, you can infer the style and period of the artists' works.
When asked about an artist you will provide relevant information based on the dataset. 
If you don't have information about a specific artist or collection, you will politely inform the user that you don't have that information.
When appropriate, you can suggest an artist from the dataset that matches the user's interests,or that of a similar style or period. You can also suggest a random artist by using the tools, when you think that's appropriate, but explain that it is a different style or period.
Only stick to the artists and the information about them you can get by using the tools, i.e. the provided dataset! Don't use other information you have in your internal dataset. Do not make up any information, if you don't know, say you don't know.
"""

In [78]:
gpt_models = [GPT_MODEL]
ollama_models = [LAMA_MODEL, DEEPSEEK_MODEL]

model = GPT_MODEL

if model in gpt_models:
    interaction_library = openai
elif model in ollama_models:
    interaction_library = ollama

def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = interaction_library.chat.completions.create(model=model, messages=messages, tools=tools)
    image_url = None
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, image_url, artist_name, title = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = interaction_library.chat.completions.create(model=model, messages=messages, tools=tools)
    
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    if image_url:
        image = get_image(image_url)

    return history, image, # artist_name, title

In [79]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.


No artist found with name: Campin
Search URL: https://collectionapi.metmuseum.org/public/collection/v1/search?artistOrCulture=true&q=Robert%20Campin
Object URL: https://collectionapi.metmuseum.org/public/collection/v1/objects/435838
Image URL: https://images.metmuseum.org/CRDImages/ep/web-large/DP-41688-001.jpg
Found artist: Robert Campin
Found 29 artists within 10 years tolerance for '1375–1444'
Selected contemporary artist: Robert Campin
Found 29 artists within 10 years tolerance for '1375-1444'
Selected contemporary artist: Francesco Pesellino
Found artist: Francesco Pesellino
Search URL: https://collectionapi.metmuseum.org/public/collection/v1/search?artistOrCulture=true&q=Francesco%20Pesellino
Object URL: https://collectionapi.metmuseum.org/public/collection/v1/objects/437276
Image URL: https://images.metmuseum.org/CRDImages/ep/web-large/DP247059.jpg
